In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.vgg16 import preprocess_input
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam

# Load dataset
photos_csv = r"C:\Users\Gurbachan\Desktop\CV_ASS2\image\photos.csv"
photos_folder = r"C:\Users\Gurbachan\Desktop\CV_ASS2\image\photos"
df = pd.read_csv(photos_csv)

# Display dataset info
df.info()
print(df.head())

# Encode labels
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)

# Build a dictionary to map photo_id to the image file paths
photo_id_to_path = {file.split('.')[0]: os.path.join(photos_folder, file) for file in os.listdir(photos_folder)}

def preprocess_image(img_path, target_size=(224, 224)):
    try:
        img = image.load_img(img_path, target_size=target_size)
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        return preprocess_input(img_array)  # Normalize the image
    except Exception as e:
        print(f"Error loading image: {img_path} - {e}")
        return None

# Preprocess images
x_images, y_labels = [], []
for _, row in df.iterrows():
    img_path = photo_id_to_path.get(row['photo_id'])
    if img_path:
        img = preprocess_image(img_path)
        if img is not None:
            x_images.append(img[0])
            y_labels.append(row['encoded_label'])

# Convert lists to numpy arrays
x_images = np.array(x_images)
y_labels = np.array(y_labels)

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x_images, y_labels, test_size=0.2, random_state=42)

# Define model architecture with Transfer Learning
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers initially
for layer in base_model.layers:
    layer.trainable = False

image_input = Input(shape=(224, 224, 3))
x = base_model(image_input, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
final_output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=image_input, outputs=final_output)

# Compile and train the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test))

# Unfreeze the last 4 layers for fine-tuning
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Compile again with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=0.00001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fine-tune the model
model.fit(x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test))

# Save the model
model.save("vgg16_transfer_learning.h5")
